# Exportação da camada Gold para CSV

Este notebook exporta as sete tabelas Gold do catálogo para CSV, com cabeçalho e separador ponto e vírgula.

Os arquivos são gravados no Unity Catalog Volume em uma pasta por tabela.
Uma execução nova sobrescreve a exportação anterior de cada tabela.

O Volume de destino deve ter acesso restrito às pessoas e serviços autorizados.

In [ ]:
import json
import os
import zoneinfo
from datetime import datetime

# 1. Caminho direto e oficial do Volume no Unity Catalog
PATH_EXPORT = "/Volumes/workspace/gold/gold_export_volume/export_csv"

# 2. Configuração de Widgets para leitura de parâmetros do Job / Workflow
dbutils.widgets.text("ambiente", "", "Ambiente de Execução (Opcional)")
dbutils.widgets.text("run_id", "", "ID do Run do Job (Opcional)")

ambiente_param = dbutils.widgets.get("ambiente").strip()
run_id_param = dbutils.widgets.get("run_id").strip()

# 3. Configura o fuso horário de Brasília/Recife (UTC-3) para a exportação
fuso_br = zoneinfo.ZoneInfo("America/Recife")

# 4. Consulta à tabela de controle (Contrato PR 54 + Fail Fast + Captura de Ambiente e Timestamp em UTC-3)
run_id_silver = None
lote_bronze_silver = None
run_id_gold_tabela = None
ambiente_tabela = None
data_processamento_gold = None

try:
    df_controle = spark.sql("""
        SELECT 
            run_id_silver, 
            lote_bronze_silver, 
            run_id_gold, 
            ambiente, 
            date_format(from_utc_timestamp(processado_em, 'America/Recife'), 'yyyy-MM-dd HH:mm:ss') AS processado_em_str
        FROM workspace.gold.controle_gold 
        WHERE status_gold = 'CONCLUIDO' 
        ORDER BY processado_em DESC 
        LIMIT 1
    """).collect()

    # Fail Fast: Se a consulta não retornar registros válidos, interrompe com erro
    if not df_controle:
        raise ValueError("Nenhum registro com 'status_gold = CONCLUIDO' foi encontrado na tabela 'workspace.gold.controle_gold'.")

    # Extração dos dados de controle
    run_id_silver = df_controle[0]["run_id_silver"]
    lote_bronze_silver = df_controle[0]["lote_bronze_silver"]
    run_id_gold_tabela = df_controle[0]["run_id_gold"]
    ambiente_tabela = df_controle[0]["ambiente"]
    data_processamento_gold = df_controle[0]["processado_em_str"]

except Exception as e:
    # Fail Fast: Interrompe a execução do Job impedindo a geração de manifesto com linhagem nula
    raise RuntimeError(f"[ERRO CRÍTICO] Falha ao consultar a tabela 'controle_gold' para validar linhagem: {e}")

# 5. Definição do Ambiente e Run ID / Lote Gold
# O ambiente do controle é obrigatório; o parâmetro não pode promovê-lo a prod.
ambiente_controle = str(ambiente_tabela or "").strip().lower()
ambiente_solicitado = ambiente_param.strip().lower()
if ambiente_controle != "prod":
    raise RuntimeError("Exportação oficial exige ambiente prod no controle_gold.")
if ambiente_solicitado and ambiente_solicitado != "prod":
    raise RuntimeError("O parâmetro ambiente da exportação deve ser prod.")
ambiente_final = "prod"

# Run ID: Prioriza Widget -> Tabela controle_gold
run_id_real = run_id_param or run_id_gold_tabela

# Fallback 1: Tentativa via Contexto de Tags do Job no Databricks (usando safeToJson)
if not run_id_real:
    try:
        context_obj = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        if hasattr(context_obj, "safeToJson"):
            context_str = context_obj.safeToJson()
        else:
            context_str = context_obj.toJson()
            
        context = json.loads(context_str)
        tags = context.get("tags", {})
        run_id_real = tags.get("multitaskParentRunId") or tags.get("runId") or tags.get("currentRunId")
    except Exception as e:
        print(f"[INFO] Contexto de Job não detectado via API: {e}")

# Fallback 2: Identificador para Execução Manual (apenas se tudo acima falhar)
if not run_id_real or run_id_real in ["run_manual_local", "execucao_manual"]:
    try:
        user_email = spark.sql("SELECT current_user()").collect()[0][0]
        user_prefix = user_email.split("@")[0]
    except Exception:
        user_prefix = "beatriz"
    
    timestamp_atual = datetime.now(fuso_br).strftime("%Y%m%dT%H%M%S")
    run_id_real = f"batch_{user_prefix}_{timestamp_atual}"



In [0]:

# Configuração do Ambiente e Volume

CATALOG = "workspace"
SCHEMA_GOLD = "gold"

# Cria o Volume no Databricks via Spark SQL usando as variáveis
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}.gold_export_volume")

# Caminho raiz para a exportação dos CSVs e do Manifesto
PATH_EXPORT = f"/Volumes/{CATALOG}/{SCHEMA_GOLD}/gold_export_volume/export_csv"

In [0]:


TABELAS_GOLD = [
    "diagnostico_receita",
    "indicadores_dec_fec",
    "fila_inspecao",
    "visao_360_uc",
    "consumo_mensal_uc",
    "consumo_receita_bairro",
    "variacao_consumo_uc"
]

print(f"Iniciando exportação de {len(TABELAS_GOLD)} tabelas Gold.")
print(f"Destino: {PATH_EXPORT}")

# Verifica todas as tabelas antes de começar, para não terminar
# com uma exportação parcial por nome incorreto ou tabela ausente.
tabelas_ausentes = [
    nome
    for nome in TABELAS_GOLD
    if not spark.catalog.tableExists(
        f"{CATALOG}.{SCHEMA_GOLD}.{nome}"
    )
]

if tabelas_ausentes:
    raise ValueError(
        f"Tabelas Gold não encontradas: {tabelas_ausentes}"
    )

for nome_tabela in TABELAS_GOLD:
    tabela_completa = f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela}"
    caminho_saida = f"{PATH_EXPORT}/{nome_tabela}"

    df = spark.table(tabela_completa)

    (
        df.write
        .format("csv")
        .mode("overwrite")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("quoteAll", "true")
        .save(caminho_saida)
    )

    print(f"[OK] {tabela_completa} -> {caminho_saida}")

print("Exportação das sete tabelas Gold concluída.")

## Manifesto da exportação oficial


### Geração do manifesto de exportação (`manifest.json`)

Antes de criar o Volume ou sobrescrever CSVs, o notebook consulta a execução mais recente com status `CONCLUIDO` em `controle_gold`. O ambiente desse registro deve ser `prod` (ignorando caixa e espaços). Ambiente ausente, `dev`, `test` ou outro valor interrompe a execução. O parâmetro opcional `ambiente`, quando preenchido, também deve ser `prod` e não substitui o ambiente registrado na Gold.

Após exportar as sete tabelas, o manifesto registra ambiente, linhagem Gold/Silver/Bronze e horários de processamento e exportação. Sua gravação ocorre no Volume autorizado, depois dos CSVs. Uma falha de escrita ainda pode deixar uma exportação parcial: a validação de ambiente não torna a publicação atômica.


In [0]:
# Capturar o horário após a gravação dos CSVs.
data_exportacao_formatada = datetime.now(fuso_br).strftime("%Y-%m-%d %H:%M:%S")

# 6. Montagem da Estrutura do Manifesto (Conforme Contrato PR 54)
manifesto = {
    "ambiente": ambiente_final,
    "lote": run_id_real,
    "run_id": run_id_real,
    "run_id_silver": run_id_silver,
    "lote_bronze_silver": lote_bronze_silver,
    "data_processamento": data_processamento_gold if data_processamento_gold else data_exportacao_formatada,
    "data_exportacao_csv": data_exportacao_formatada,
    "tabelas_exportadas": [
        "diagnostico_receita",
        "indicadores_dec_fec",
        "fila_inspecao",
        "visao_360_uc",
        "consumo_mensal_uc",
        "consumo_receita_bairro",
        "variacao_consumo_uc"
    ],
    "responsavel": "Equipe de Engenharia de Dados (Gold)"
}

# 7. Gravação do arquivo manifest.json no Volume indicado
os.makedirs(PATH_EXPORT, exist_ok=True)
caminho_manifesto = os.path.join(PATH_EXPORT, "manifest.json")

with open(caminho_manifesto, "w", encoding="utf-8") as f:
    json.dump(manifesto, f, indent=4, ensure_ascii=False)

print(f"[OK] Manifesto gerado e validado com sucesso!")
print(f" Ambiente: {manifesto['ambiente']}")
print(f" Lote / Run ID Gold: {run_id_real}")
print(f" Run ID Silver: {run_id_silver}")
print(f" Lote Bronze/Silver: {lote_bronze_silver}")
print(f" Data do Processamento (Gold): {manifesto['data_processamento']}")
print(f" Data da Exportação (CSV): {manifesto['data_exportacao_csv']}")
print(f" Arquivo salvo em: {caminho_manifesto}")

---

## 8. Convenção de Linhagem e Rastreabilidade (PR-54)

Para garantir a auditabilidade e o rastreamento ponta a ponta dos dados no manifesto de exportação (`manifest.json`), adotamos as seguintes convenções de identificação entre as camadas:

### 🔹 Camada Silver (`lote_bronze_silver` vs `run_id_silver`)
- **Natureza:** Incremental e de saneamento de dados.
- **`lote_bronze_silver`:** Identificador do lote físico original vindo da camada Bronze (preserva a "etiqueta de nascimento" dos dados, ex: `20261022T194649Z_d35f6c5e`).
- **`run_id_silver`:** Identificador único da execução do pipeline da Silver que processou ou reprocessou aquele lote (ex: `20261028T022140Z_b5d60d92`).

### 🔹 Camada Gold (`lote` / `run_id`)
- **Natureza:** Visões consolidadas e agregadas para consumo analítico.
- **`lote` == `run_id`:** Na camada Gold, o lote é a própria execução do Job de consolidação (`run_id`). Como a Gold recomputa as métricas a partir do estado atual da Silver, a rodada da Gold representa o snapshot/lote completo dos indicadores gerados.
- **Formato do ID Gold:** Gerado no padrão `GOLD_<TIMESTAMP_UTC>_<HASH_8CHAR>` (ex: `GOLD_20261028T024507Z_7f5f4fda`).
- **Rastreabilidade:** Cada manifesto da Gold mapeia exatamente qual `run_id_silver` e `lote_bronze_silver` serviram de insumo para a geração daquela versão dos dados.

###  Exemplo da Estrutura do `manifest.json`
```json
{
    "ambiente": "prod",
    "lote": "GOLD_20261028T024507Z_7f5f4fda",
    "run_id": "GOLD_20261028T024507Z_7f5f4fda",
    "run_id_silver": "20261028T022140Z_b5d60d92",
    "lote_bronze_silver": "20261022T194649Z_d35f6c5e",
    "data_processamento": "2026-10-27 23:45:07",
    "data_exportacao_csv": "2026-10-27 23:58:23",
    "responsavel": "Equipe de Engenharia de Dados (Gold)"
}